# 04_reference_data_silver

## Purpose

Transform Bronze asset reference records into a cleaned and reusable Silver lookup table.

This notebook standardizes asset identifiers, region values, and asset types, removes duplicates, and writes a stable reference table for later joins.

In [0]:
import sys

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from pyspark.sql import functions as F
from validation.silver_checks import print_basic_quality_summary

In [0]:
import sys

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from pyspark.sql import functions as F
from validation.silver_checks import print_basic_quality_summary

In [0]:
catalog = "vattenfall_dev"

bronze_table = f"{catalog}.raw.bronze_asset_reference"
silver_table = f"{catalog}.refined.silver_asset_reference"

print("Source Bronze table:", bronze_table)
print("Target Silver table:", silver_table)

In [0]:
bronze_reference_df = spark.table(bronze_table)

before_count = bronze_reference_df.count()

print("Rows before cleaning:", before_count)
display(bronze_reference_df.limit(20))

In [0]:
silver_reference_df = (
    bronze_reference_df
    .withColumn("asset_id", F.upper(F.trim(F.col("asset_id"))))
    .withColumn("asset_name", F.trim(F.col("asset_name")))
    .withColumn("region", F.upper(F.trim(F.col("region"))))
    .withColumn("asset_type", F.upper(F.trim(F.col("asset_type"))))
    .filter(F.col("asset_id").isNotNull())
    .filter(F.col("region").isNotNull())
    .dropDuplicates(["asset_id"])
)

after_count = silver_reference_df.count()

print("Rows after cleaning:", after_count)
print("Rows removed:", before_count - after_count)

display(silver_reference_df.limit(20))

In [0]:
(
    silver_reference_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print("Written Silver table:", silver_table)

In [0]:
silver_df = spark.table(silver_table)

print_basic_quality_summary(silver_df, silver_table)

display(silver_df.limit(20))

In [0]:
print("Asset type values:")
silver_df.select("asset_type").distinct().show(truncate=False)

print("Region values:")
silver_df.select("region").distinct().show(truncate=False)

print("Null check for key Silver fields:")
for column_name in ["asset_id", "asset_name", "region", "asset_type"]:
    null_count = silver_df.filter(f"{column_name} IS NULL").count()
    print(column_name, "nulls:", null_count)

print("Duplicate asset_id count:")
duplicate_count = (
    silver_df
    .groupBy("asset_id")
    .count()
    .filter("count > 1")
    .count()
)

print("duplicate asset_id rows:", duplicate_count)